In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
df = pd.read_csv("data/cardataset.csv")
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [3]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.head()

,make,model,year,engine_fuel_type,engine_hp,engine_cylinders,transmission_type,driven_wheels,number_of_doors,market_category,vehicle_size,vehicle_style,highway_mpg,city_mpg,popularity,msrp
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [4]:
df.dtypes

make                     str
model                    str
year                   int64
engine_fuel_type         str
engine_hp            float64
engine_cylinders     float64
transmission_type        str
driven_wheels            str
number_of_doors      float64
market_category          str
vehicle_size             str
vehicle_style            str
highway_mpg            int64
city_mpg               int64
popularity             int64
msrp                   int64
dtype: object

In [5]:
columns_strings = df.dtypes[df.dtypes == 'object'].index.to_list()
print(columns_strings)
#for col in columns_strings:
#    df[col] = df[col].str.lower().str.replace(' ', '_')
#
#df.head()

[]


In [ ]:
df_eda = df[['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity', 'year', 'msrp']].copy()

df_eda['age'] = 2017 - df_eda['year']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(np.log1p(df_eda['msrp']), bins=40, color='#2a9d8f', edgecolor='white')
axes[0, 0].set_title('Distribución del precio (log1p)')
axes[0, 0].set_xlabel('log1p(msrp)')
axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].scatter(df_eda['age'], df_eda['msrp'], alpha=0.2, s=10, color='#e76f51')
axes[0, 1].set_title('Precio vs antigüedad')
axes[0, 1].set_xlabel('age')
axes[0, 1].set_ylabel('msrp')

axes[1, 0].scatter(df_eda['engine_hp'], df_eda['msrp'], alpha=0.2, s=10, color='#264653')
axes[1, 0].set_title('Precio vs horsepower')
axes[1, 0].set_xlabel('engine_hp')
axes[1, 0].set_ylabel('msrp')

corr_cols = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity', 'age', 'msrp']
corr = df_eda[corr_cols].corr()
im = axes[1, 1].imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(corr_cols)))
axes[1, 1].set_xticklabels(corr_cols, rotation=45, ha='right')
axes[1, 1].set_yticks(range(len(corr_cols)))
axes[1, 1].set_yticklabels(corr_cols)
axes[1, 1].set_title('Correlación entre variables')
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

fig.tight_layout()
plt.show()

In [ ]:
base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']

def prepare_X(df: pd.DataFrame) -> np.typing.NDArray:
    df = df.copy()

    df['age'] = 2017 - df['year']
    features = base + ['age']

    df_num = df[features]
    df_num = df_num.fillna(0)

    X = df_num.to_numpy()

    return X

def train_test_split(df: pd.DataFrame, train_size: float = 0.6, val_size: float = 0.2, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n_train = int(n * train_size)
    n_val = int(n * val_size)

    df_train = df.iloc[idx[:n_train]].copy()
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
    df_test = df.iloc[idx[n_train + n_val:]].copy()

    return df_train, df_val, df_test

def predict(X: np.typing.NDArray, w: np.typing.NDArray, b: float) -> np.typing.NDArray:
    return X @ w + b

def rmse(y_true: np.typing.NDArray, y_pred: np.typing.NDArray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

df_model = df[['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity', 'year', 'msrp']].copy()
df_train, df_val, df_test = train_test_split(df_model)

X_train = prepare_X(df_train)
X_val = prepare_X(df_val)
X_test = prepare_X(df_test)

y_train = np.log1p(df_train['msrp'].values)
y_val = np.log1p(df_val['msrp'].values)
y_test = np.log1p(df_test['msrp'].values)

X_train_1 = np.column_stack([np.ones(X_train.shape[0]), X_train])
w_full, *_ = np.linalg.lstsq(X_train_1, y_train, rcond=None)
b = float(w_full[0])
w = w_full[1:]

y_val_pred = predict(X_val, w, b)
val_rmse = rmse(y_val, y_val_pred)

print(f'Validation RMSE: {val_rmse:.4f}')
print(f'Bias: {b:.4f}')
print('Weights:', w)

In [ ]:
y_test_pred = predict(X_test, w, b)
test_rmse = rmse(y_test, y_test_pred)

sample = df_test.iloc[[0]]
sample_pred = float(np.expm1(predict(prepare_X(sample), w, b))[0])
sample_actual = float(sample['msrp'].iloc[0])

print(f'Test RMSE: {test_rmse:.4f}')
print(f'Predicted MSRP: ${sample_pred:,.0f}')
print(f'Actual MSRP: ${sample_actual:,.0f}')

In [ ]:
y_test_pred_price = np.expm1(y_test_pred)
y_test_price = np.expm1(y_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_price, y_test_pred_price, alpha=0.2, s=10, color='#457b9d')
min_price = float(min(y_test_price.min(), y_test_pred_price.min()))
max_price = float(max(y_test_price.max(), y_test_pred_price.max()))
axes[0].plot([min_price, max_price], [min_price, max_price], '--', color='#e63946')
axes[0].set_title('Predicho vs real')
axes[0].set_xlabel('Precio real')
axes[0].set_ylabel('Precio predicho')

residuals = y_test_price - y_test_pred_price
axes[1].hist(residuals, bins=40, color='#2a9d8f', edgecolor='white')
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Precio real - predicho')
axes[1].set_ylabel('Frecuencia')

fig.tight_layout()
plt.show()